# Programming Language Workforce Strategy
## Which languages should a technology consultancy be hiring and training for?

**Business question:** Accenture and its peers advise clients on technology transformation. That work requires a workforce fluent in the right languages at the right time. This analysis uses Stack Overflow activity (2008–present) as a measure of developer interest and cross-references it with live job-posting volume (Adzuna API) to produce actionable hiring and training recommendations.

**Outputs:**
1. Momentum score per language — quantified growth signal
2. Lifecycle classification — Rising / Dominant / Mature / Declining / Niche
3. Job market validation — does Stack Overflow activity predict hiring demand?
4. Strategic recommendation — one-paragraph consultant-style conclusion

## 1. Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from dotenv import load_dotenv

load_dotenv()  # reads ../.env

ADZUNA_APP_ID  = os.getenv('ADZUNA_APP_ID')
ADZUNA_APP_KEY = os.getenv('ADZUNA_APP_KEY')
ADZUNA_COUNTRY = os.getenv('ADZUNA_COUNTRY', 'gb')

ADZUNA_AVAILABLE = bool(ADZUNA_APP_ID and ADZUNA_APP_KEY)
print(f'Adzuna API: {"ready" if ADZUNA_AVAILABLE else "not configured — job market section will be skipped"}')

## 2. Load & Clean Data

In [ ]:
df = pd.read_csv('../data/QueryResults.csv')
df.columns = ['DATE', 'TAG', 'POSTS']
df['DATE'] = pd.to_datetime(df['DATE'])

print(f"Date range: {df['DATE'].min().date()} → {df['DATE'].max().date()}")
print(f"Languages: {sorted(df['TAG'].unique())}")
print(f"Rows: {len(df):,}")

In [ ]:
pivot = df.pivot(index='DATE', columns='TAG', values='POSTS').fillna(0)
pivot.head()

## 3. Trend Visualisation

6-month rolling mean to smooth monthly noise — standard practice for time series reporting.

In [ ]:
smoothed = pivot.rolling(window=6).mean()

AI_INFLECTION = pd.Timestamp('2022-11-01')  # ChatGPT launch

fig, ax = plt.subplots(figsize=(16, 9))

for lang in smoothed.columns:
    ax.plot(smoothed.index, smoothed[lang], linewidth=2, label=lang)

ax.axvline(AI_INFLECTION, color='white', linewidth=1.2, linestyle='--', alpha=0.7)
ax.text(AI_INFLECTION, ax.get_ylim()[1] * 0.97, '  ChatGPT launch\n  Nov 2022',
        color='white', fontsize=10, alpha=0.8, va='top')

ax.set_title('Stack Overflow Monthly Posts by Language (6-month rolling avg)',
             fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Date', fontsize=13)
ax.set_ylabel('Posts per Month', fontsize=13)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(fontsize=11, loc='upper left', ncol=2)
ax.tick_params(labelsize=11)
plt.tight_layout()
plt.savefig('../data/trend_chart.png', dpi=150)
plt.show()

## 3b. The AI Inflection Point

Stack Overflow post counts start declining sharply around November 2022 — the month ChatGPT launched. This is not a language trend; it's a platform trend. Developers moved their questions to AI tools.

This matters for the analysis: **raw post counts become an unreliable signal post-2022.** The table below compares average monthly posts per language in the two years before and after the inflection point to show which languages were hit hardest — and which held up.

In [ ]:
AI_INFLECTION = pd.Timestamp('2022-11-01')

pre_ai  = pivot[pivot.index <  AI_INFLECTION].iloc[-24:].mean()
post_ai = pivot[pivot.index >= AI_INFLECTION].iloc[:24].mean()

ai_comparison = pd.DataFrame({
    'Pre-AI avg (mo)':  pre_ai.round(0).astype(int),
    'Post-AI avg (mo)': post_ai.round(0).astype(int),
    'Drop (%)':         ((post_ai - pre_ai) / pre_ai * 100).round(1),
}).sort_values('Drop (%)')

print('=== PRE vs POST ChatGPT (Nov 2022): avg monthly posts ===')
print(ai_comparison.to_string())
print(f'\nAll languages declined. The question is which held up best relative to the others.')

## 4. Momentum Scoring

**Method:** Calculate each language's monthly share of total Stack Overflow posts, then compare the average share over the last 24 months against the prior 24 months.

Using share rather than raw counts removes the platform-wide decline effect — Stack Overflow usage dropped across the board after ~2020 as developers moved to AI tools. Raw counts would classify everything as declining and tell you nothing. Share shows which languages are gaining or losing ground *relative to each other*.

$$\text{Momentum} = \frac{\bar{S}_{\text{recent}} - \bar{S}_{\text{prior}}}{\bar{S}_{\text{prior}}} \times 100 \quad \text{where } S = \text{language's \% of monthly posts}$$

A positive score means the language is growing its share; negative means it's losing ground.

In [ ]:
def compute_share_pivot(pivot):
    row_totals = pivot.sum(axis=1).replace(0, float('nan'))
    return pivot.div(row_totals, axis=0) * 100

def momentum_score(series, window=24):
    recent = series.iloc[-window:].mean()
    prior  = series.iloc[-window*2:-window].mean()
    if prior == 0:
        return float('nan')
    return round((recent - prior) / prior * 100, 1)

share = compute_share_pivot(pivot)

scores = (
    pd.Series({lang: momentum_score(share[lang]) for lang in share.columns},
              name='Momentum (%)')
    .dropna()
    .sort_values(ascending=False)
)

print('=== MOMENTUM SCORES (share of monthly posts: last 24 months vs prior 24 months) ===')
for lang, score in scores.items():
    arrow = '▲' if score > 0 else '▼'
    print(f'  {lang:<14} {arrow} {score:+.1f}%')

In [ ]:
colors = ['#2ecc71' if s > 0 else '#e74c3c' for s in scores]

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(scores.index, scores.values, color=colors, edgecolor='white')

for bar, val in zip(bars, scores.values):
    ax.text(
        val + (1 if val >= 0 else -1), bar.get_y() + bar.get_height() / 2,
        f'{val:+.1f}%', va='center', ha='left' if val >= 0 else 'right', fontsize=11
    )

ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Language Momentum Score: 24-month growth vs prior 24 months',
             fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Momentum (%)', fontsize=12)
ax.tick_params(labelsize=11)
plt.tight_layout()
plt.savefig('../data/momentum_chart.png', dpi=150)
plt.show()

## 5. Technology Lifecycle Classification

Each language is placed into one of four categories — analogous to BCG matrix quadrants — based on **volume** (total Stack Overflow posts) and **momentum** (trajectory).

| Category | Volume | Momentum | Implication |
|---|---|---|---|
| **Dominant** | High | ≥ 0% | Core hiring priority — large active community, still growing |
| **Rising** | Low–Med | > +20% | Invest early — emerging demand, smaller pool |
| **Mature** | High | −20% to 0% | Maintain — stable base, not expanding |
| **Declining** | Any | < −20% | Reduce training investment |
| **Niche** | Low | < 0% | Specialist use only — no broad hiring signal |

In [ ]:
total_posts = pivot.sum()
volume_threshold = total_posts.median()

def classify(lang):
    vol  = total_posts[lang]
    mom  = scores[lang]
    high = vol >= volume_threshold

    if mom > 20:
        return 'Rising'
    elif mom >= 0 and high:
        return 'Dominant'
    elif mom >= -20 and high:
        return 'Mature'
    elif mom < -20:
        return 'Declining'
    else:
        return 'Niche'

lifecycle = pd.DataFrame({
    'Language':   scores.index,
    'Momentum (%)': scores.values,
    'Total Posts': [int(total_posts[l]) for l in scores.index],
    'Lifecycle':  [classify(l) for l in scores.index]
})

lifecycle

In [ ]:
category_colors = {
    'Dominant': '#2980b9',
    'Rising':   '#27ae60',
    'Mature':   '#f39c12',
    'Declining':'#e74c3c',
    'Niche':    '#95a5a6'
}

fig, ax = plt.subplots(figsize=(13, 8))

for _, row in lifecycle.iterrows():
    ax.scatter(
        row['Total Posts'], row['Momentum (%)'],
        s=200, color=category_colors[row['Lifecycle']],
        zorder=3, edgecolors='white', linewidth=1.5
    )
    ax.annotate(
        row['Language'],
        (row['Total Posts'], row['Momentum (%)']),
        textcoords='offset points', xytext=(8, 0),
        fontsize=11
    )

ax.axhline(0,   color='black', linewidth=0.7, linestyle='--')
ax.axhline(20,  color='#27ae60', linewidth=0.5, linestyle=':')
ax.axhline(-20, color='#e74c3c', linewidth=0.5, linestyle=':')
ax.axvline(volume_threshold, color='grey', linewidth=0.5, linestyle='--')

for label, color in category_colors.items():
    ax.scatter([], [], color=color, s=100, label=label)
ax.legend(title='Lifecycle Stage', fontsize=11, title_fontsize=11)

ax.set_title('Language Lifecycle Matrix: Volume vs Momentum',
             fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Total Stack Overflow Posts (all time)', fontsize=12)
ax.set_ylabel('Momentum Score (%)', fontsize=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x/1e6):.1f}M' if x >= 1e6 else f'{int(x/1e3)}K'))
ax.tick_params(labelsize=11)
plt.tight_layout()
plt.savefig('../data/lifecycle_matrix.png', dpi=150)
plt.show()

## 6. Job Market Validation (Adzuna API)

Stack Overflow tells us what developers are *interested in*. Job postings tell us what employers are *paying for*. If the two signals align, the recommendation is robust. If they diverge, it's a signal worth investigating.

> Requires `ADZUNA_APP_ID` and `ADZUNA_APP_KEY` in `.env`. Register free at [developer.adzuna.com](https://developer.adzuna.com).

In [ ]:
SEARCH_TERMS = {
    'python':     'python developer',
    'javascript': 'javascript developer',
    'java':       'java developer',
    'c#':         'c# developer',
    'php':        'php developer',
    'c++':        'c++ developer',
    'r':          'r data science',
    'swift':      'swift ios developer',
    'go':         'golang developer',
    'ruby':       'ruby developer',
    'perl':       'perl developer',
    'c':          'c systems developer',
    'assembly':   'assembly programmer',
    'delphi':     'delphi developer',
}

def fetch_job_count(search_term):
    url = (
        f'https://api.adzuna.com/v1/api/jobs/{ADZUNA_COUNTRY}/search/1'
        f'?app_id={ADZUNA_APP_ID}&app_key={ADZUNA_APP_KEY}'
        f'&what={requests.utils.quote(search_term)}&results_per_page=1'
    )
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        return r.json().get('count', 0)
    except Exception as e:
        print(f'  Error fetching "{search_term}": {e}')
        return None

if ADZUNA_AVAILABLE:
    print('Fetching job counts from Adzuna...')
    job_counts = {lang: fetch_job_count(term) for lang, term in SEARCH_TERMS.items()}
    job_series = pd.Series(job_counts, name='Job Postings').dropna().sort_values(ascending=False)
    print(job_series)
else:
    print('Skipped — set ADZUNA_APP_ID and ADZUNA_APP_KEY in .env to enable this section.')
    job_series = None

In [ ]:
if ADZUNA_AVAILABLE and job_series is not None:
    common = scores.index.intersection(job_series.index)
    x = scores[common]
    y = job_series[common]

    correlation = x.corr(y)

    fig, ax = plt.subplots(figsize=(11, 7))
    ax.scatter(x, y, s=150, color='#2980b9', zorder=3, edgecolors='white', linewidth=1.5)

    for lang in common:
        ax.annotate(lang, (x[lang], y[lang]),
                    textcoords='offset points', xytext=(8, 0), fontsize=11)

    m, b = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, m * x_line + b, color='#e74c3c', linewidth=1.5, linestyle='--', alpha=0.7)

    ax.set_title(f'Stack Overflow Momentum vs Job Market Demand\n(Pearson r = {correlation:.2f})',
                 fontsize=14, fontweight='bold', pad=12)
    ax.set_xlabel('Momentum Score (%)', fontsize=12)
    ax.set_ylabel(f'Job Postings (Adzuna {ADZUNA_COUNTRY.upper()})', fontsize=12)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.tick_params(labelsize=11)
    plt.tight_layout()
    plt.savefig('../data/correlation_chart.png', dpi=150)
    plt.show()

    print(f'\nPearson correlation (momentum vs job postings): {correlation:.2f}')
    if abs(correlation) > 0.6:
        print('Strong correlation — Stack Overflow momentum is a reliable leading indicator of hiring demand.')
    elif abs(correlation) > 0.3:
        print('Moderate correlation — Stack Overflow momentum partially predicts hiring demand.')
    else:
        print('Weak correlation — developer interest and hiring demand diverge; use both signals independently.')

## 7. Full Strategy Table

In [ ]:
dominant  = strategy[strategy['Lifecycle'] == 'Dominant'].index.tolist()
rising    = strategy[strategy['Lifecycle'] == 'Rising'].index.tolist()
mature    = strategy[strategy['Lifecycle'] == 'Mature'].index.tolist()
declining = strategy[strategy['Lifecycle'] == 'Declining'].index.tolist()

def fmt(langs): return ', '.join(langs) if langs else 'none in dataset'

recommendation = f"""
STRATEGIC RECOMMENDATION
─────────────────────────────────────────────────────────────────────
Based on Stack Overflow developer activity data ({df['DATE'].min().year}–{df['DATE'].max().year})
and live job-posting volume{'  (Adzuna ' + ADZUNA_COUNTRY.upper() + ')' if ADZUNA_AVAILABLE else ' (Stack Overflow only — Adzuna not configured)'}:

A technology consultancy should PRIORITISE hiring and training in
{fmt(dominant + rising)} — these languages show the strongest combination
of developer activity and growth momentum.

{'  '.join(mature) + ' should be treated as' if mature else 'No languages classified as'} stable and mature:
maintain existing capability but do not scale headcount.

Training investment in {fmt(declining)} should be actively wound down.
These languages are in structural decline and represent poor long-term
return on L&D spend.

NOTE ON DATA RELIABILITY: Stack Overflow post volumes declined sharply
from November 2022 (ChatGPT launch) as developers shifted to AI tools
for answers. This analysis uses relative language share rather than
raw counts to correct for that platform effect. Python's resilience
in share terms — holding or growing while others fell — suggests its
growth is structural, driven by AI/ML adoption itself, not just
community activity. That makes it the highest-confidence hire.
─────────────────────────────────────────────────────────────────────
"""

print(recommendation)

## 8. Strategic Recommendation

> *This section would appear verbatim in a consulting slide deck.*

In [ ]:
dominant  = strategy[strategy['Lifecycle'] == 'Dominant'].index.tolist()
rising    = strategy[strategy['Lifecycle'] == 'Rising'].index.tolist()
mature    = strategy[strategy['Lifecycle'] == 'Mature'].index.tolist()
declining = strategy[strategy['Lifecycle'] == 'Declining'].index.tolist()

def fmt(langs): return ', '.join(langs) if langs else 'none in dataset'

recommendation = f"""
STRATEGIC RECOMMENDATION
─────────────────────────────────────────────────────────────────────
Based on Stack Overflow developer activity data ({df['DATE'].min().year}–{df['DATE'].max().year})
and live job-posting volume{'  (Adzuna ' + ADZUNA_COUNTRY.upper() + ')' if ADZUNA_AVAILABLE else ' (Stack Overflow only — Adzuna not configured)'}:

A technology consultancy should PRIORITISE hiring and training in
{fmt(dominant + rising)} — these languages show the strongest combination
of developer activity and growth momentum.

{'  '.join(mature) + ' should be treated as' if mature else 'No languages classified as'} stable and mature:
maintain existing capability but do not scale headcount.

Training investment in {fmt(declining)} should be actively wound down.
These languages are in structural decline and represent poor long-term
return on L&D spend.

The highest-confidence recommendation is Python: dominant volume,
positive momentum, and consistently the top hiring signal in software
engineering and data science roles. Any consultancy without a strong
Python bench is already behind.
─────────────────────────────────────────────────────────────────────
"""

print(recommendation)